# 01 Human and Crowd Reconstruction

## 1. Introduction and scientific limitations
Identity preservation takes priority over sharpness. Manual observations and synthetic ground truth are not identity proof. Source-scale tiers are provisional and require experimental calibration. Missing landmark, embedding, anatomy and human judgments require visual review.

Learned SR synthesizes detail and is classified Inference; the explicit Lanczos stand-in is Interpolation. Region microcontrast is simulated enhancement, not recovered texture. TIFF16 and half EXR add encoding precision, not sensor measurements; linear EXR remains display-derived. Selected SR and detector weights may download at runtime. Generative face replacement and sensor simulation are disabled. Grain is optional, explicitly labelled Simulation.

## 2. Runtime setup
Use `quick` for a reduced output, `standard` for the configured 50 MP target, or `research` for the same target with the detailed comparisons and metrics supplied to every run. No extra ablations are implemented for research yet. Runtime depends on hardware. Start with quick: multiple target-sized arrays are held in memory.

For Colab, supply `PROJECT_DIR` or clone `REPO_REF`; private repositories use the Colab secret `GITHUB_TOKEN`. Set `MOUNT_DRIVE` and output paths to preserve masters. Local setup uses the existing checkout.

In [ ]:
# @title Human run parameters
REPO_URL = "https://github.com/RDC28/experiment_50MP.git"  # @param {type:"string"}
REPO_REF = "main"  # @param {type:"string"}
PROJECT_DIR = ""  # @param {type:"string"}
MOUNT_DRIVE = False  # @param {type:"boolean"}
OUTPUT_ROOT = ""  # @param {type:"string"}
REPORTS_ROOT = ""  # @param {type:"string"}
INPUTS_ROOT = ""  # @param {type:"string"}
CONFIG_FILE = "configs/human_default.yaml"  # @param {type:"string"}
INPUT_MODE = "upload"  # @param ["upload", "path", "synthetic"]
INPUT_PATH = ""  # @param {type:"string"}
RUNTIME_PRESET = "quick"  # @param ["quick", "standard", "research"]
TARGET_MEGAPIXELS = 50.0  # @param {type:"number"}
QUICK_TARGET_MEGAPIXELS = 0.1536  # @param {type:"number"}
FACE_DETECTOR = "yunet"  # @param ["yunet", "haar", "none"]
MANUAL_FACE_BOXES = []  # source-pixel [x, y, width, height, optional confidence]
PRIMARY_SUBJECT_INDEX = None  # None selects highest source tier, then largest face
# None keeps the YAML value; accepted ranges and warnings appear in section 7.
FACE_ENHANCEMENT_STRENGTH = None
HAIR_ENHANCEMENT_STRENGTH = None
FABRIC_DETAIL_STRENGTH = None
IDENTITY_THRESHOLDS = {}  # advanced overrides of documented identity thresholds
REGION_OVERRIDES = {}  # advanced overrides of documented region parameters
# Skin smoothing is bounded and capped by face enhancement strengths.
BACKGROUND_FACE_POLICY = "preserve"
SKIN_TEXTURE_SMOOTHING = 0.0
SR_BACKEND = "swinir"  # @param ["swinir", "realesrgan", "lanczos_stand_in"]
SR_DEVICE = "auto"  # @param ["auto", "cuda", "mps", "cpu"]
SR_PRECISION = "auto"  # @param ["auto", "float32", "float16"]
SR_TILE_SIZE = 0  # @param {type:"integer"}
SR_TILE_OVERLAP = 64  # @param {type:"integer"}
WEIGHTS_DIR = ""  # @param {type:"string"}
PRESERVE_SOURCE_STRUCTURE = True  # @param {type:"boolean"}
GENERAL_SR_STRENGTH = 1.0
EDGE_SHARPENING = 0.0
GRAIN = 0.0
TONAL_CONTRAST = 0.0
COLOR_BALANCE = [1.0, 1.0, 1.0]

## 3. Dependency installation
Install the shared package from the selected checkout. Credentials remain in the local request environment. Learned SR selects the `[sr]` extra (PyTorch and spandrel), matching notebook 00; the explicit Lanczos stand-in uses the base package. Use Colab for inference.

In [ ]:
import base64
import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

NOTEBOOK_VERSION = "01_Human_Crowd_50MP/1.0"
COLAB_CLONE = Path("/content/experiment_50MP")
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def find_project() -> Path:
    if PROJECT_DIR:
        return Path(PROJECT_DIR).expanduser().resolve()
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pipeline.py").exists():
            return candidate.resolve()
    return COLAB_CLONE


def git(*args: str) -> None:
    env = dict(os.environ)
    token = ""
    if IN_COLAB:
        try:
            from google.colab import userdata

            token = userdata.get("GITHUB_TOKEN") or ""
        except Exception:
            token = ""
    if token:
        credential = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(
            GIT_CONFIG_COUNT="1",
            GIT_CONFIG_KEY_0="http.extraHeader",
            GIT_CONFIG_VALUE_0=f"Authorization: Basic {credential}",
        )
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"git failed (is GITHUB_TOKEN set for this private repository?): {result.stderr}")


PROJECT = find_project()
if not (PROJECT / "pyproject.toml").exists():
    git("clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(PROJECT))
elif PROJECT == COLAB_CLONE:
    git("-C", str(PROJECT), "fetch", "--depth", "1", REPO_URL, REPO_REF)
    git("-C", str(PROJECT), "checkout", "--detach", "FETCH_HEAD")
spec = importlib.util.find_spec("synthetic_raw_lab")
missing_sr = SR_BACKEND != "lanczos_stand_in" and importlib.util.find_spec("spandrel") is None
if spec is None or Path(spec.origin).resolve().parent != PROJECT / "src" or missing_sr:
    target = f"{PROJECT}[sr]" if SR_BACKEND != "lanczos_stand_in" else str(PROJECT)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", target], check=True)
    importlib.invalidate_caches()
for name in [m for m in sys.modules if m.partition(".")[0] == "synthetic_raw_lab"]:
    del sys.modules[name]

import synthetic_raw_lab  # noqa: E402
from synthetic_raw_lab import environment  # noqa: E402

print(f"project: {PROJECT} (Git: {environment.git_revision(PROJECT)})")
print(f"synthetic_raw_lab {synthetic_raw_lab.__version__} from {Path(synthetic_raw_lab.__file__).parent}")

## 4. GPU and environment inspection
Learned SR supports automatic device selection, CUDA, MPS or CPU; use a Colab GPU runtime. SwinIR uses float32; supported Real-ESRGAN networks can use float16 on CUDA. Tile size 0 selects memory-aware tiling; overlap defaults to 64 source pixels. Cache weights on Drive with WEIGHTS_DIR.

In [ ]:
from IPython.display import Image, Markdown, display

from synthetic_raw_lab import environment, human_config, ingestion, pipeline, test_images
from synthetic_raw_lab.human_run import run_human_crowd

config_path = Path(CONFIG_FILE)
if not config_path.is_absolute():
    config_path = PROJECT / config_path
CONFIG = human_config.load_config(config_path)
paths = {"output_root": OUTPUT_ROOT, "reports_root": REPORTS_ROOT, "inputs_root": INPUTS_ROOT}
CONFIG = CONFIG.replace(
    sr_backend=SR_BACKEND,
    sr_device=SR_DEVICE,
    sr_precision=SR_PRECISION,
    sr_tile_size=SR_TILE_SIZE,
    sr_tile_overlap=SR_TILE_OVERLAP,
    sr_weights_dir=WEIGHTS_DIR,
    preserve_source_structure=PRESERVE_SOURCE_STRUCTURE,
    runtime_preset=RUNTIME_PRESET,
    target_megapixels=TARGET_MEGAPIXELS,
    quick_target_megapixels=QUICK_TARGET_MEGAPIXELS,
    **{key: value for key, value in paths.items() if value},
)
INPUTS_DIR, OUTPUT_DIR, REPORTS_DIR = pipeline.resolve_roots(CONFIG, PROJECT)
display(environment.inspect_runtime(OUTPUT_DIR))

## 5. Input upload
Choose one upload in Colab or a local path. Synthetic mode creates the small deterministic 384 x 256 crowd fixture with all three source-scale tiers and uses its ground-truth boxes as manual observations. Real images use YuNet inference by default (weights download at runtime and are SHA-256 checked, see MODELS.md); optional Haar works only on OpenCV builds that bundle its XML. Manual source-pixel boxes take precedence over overlapping detections. Detectors can miss faces; inspect the observations and add boxes. Detection is never identity proof.

In [ ]:
from dataclasses import astuple

from PIL import Image as PILImage

STAGING = OUTPUT_DIR / "_staging"
if INPUT_MODE == "synthetic":
    pixels, faces = test_images.make_crowd_test_image()
    STAGING.mkdir(parents=True, exist_ok=True)
    SOURCE = STAGING / "synthetic_crowd.png"
    PILImage.fromarray(pixels).save(SOURCE)
    boxes = [list(astuple(face)) for face in faces]
elif INPUT_MODE == "path":
    SOURCE = Path(INPUT_PATH).expanduser()
    boxes = MANUAL_FACE_BOXES
elif INPUT_MODE == "upload":
    if not IN_COLAB:
        raise RuntimeError("Use path or synthetic mode outside Colab")
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one image")
    name, data = next(iter(uploaded.items()))
    STAGING.mkdir(parents=True, exist_ok=True)
    SOURCE = STAGING / Path(name).name
    SOURCE.write_bytes(data)
    boxes = MANUAL_FACE_BOXES
else:
    raise ValueError(f"Unknown input mode: {INPUT_MODE}")
CONFIG = CONFIG.replace(
    face_detector=FACE_DETECTOR, manual_face_boxes=boxes, primary_subject_index=PRIMARY_SUBJECT_INDEX if boxes else None
)
print(f"Source: {SOURCE}; manual observations: {len(boxes)}")

## 6. Input validation
Preserve a source copy and inspect decode, color interpretation, dimensions and warnings before reconstruction.

In [ ]:
source = ingestion.ingest(SOURCE, INPUTS_DIR, CONFIG.input_category, source_color_space=CONFIG.source_color_space)
display(source.validation)
for warning in source.warnings:
    print("WARNING:", warning)
del source

## 7. Experiment configuration
Face, hair and fabric strengths accept 0 to 1; higher values risk halos and appearance changes. Identity strictness uses the documented threshold dictionary rather than a misleading universal slider. Background faces are preserved with zero default enhancement. Region masks are estimates.

General SR strength blends Lanczos (0) with the selected upscaler (1, default); the explicit Lanczos stand-in yields identical candidates. Edge sharpening (0 to 1, default 0) applies bounded unsharp detail outside observed face boxes. High strengths can amplify halos and artifacts. Both have independent provenance stages. Skin texture smoothing (0 to 1, default 0) applies a bounded bilateral filter weighted by estimated skin, capped inside complete face boxes by the corresponding face strength. High values can erase natural texture. Every implemented key is documented below with default, range, effect and risk.

In [ ]:
if BACKGROUND_FACE_POLICY != "preserve":
    raise ValueError("BACKGROUND_FACE_POLICY must be preserve; face restoration is prohibited")
regions = CONFIG.to_dict()["regions"]
regions.update(REGION_OVERRIDES)
for key, value in {
    "primary_face": FACE_ENHANCEMENT_STRENGTH,
    "hair": HAIR_ENHANCEMENT_STRENGTH,
    "clothing": FABRIC_DETAIL_STRENGTH,
}.items():
    if value is not None:
        regions[key] = value
identity = CONFIG.to_dict()["identity"]
identity.update(IDENTITY_THRESHOLDS)
CONFIG = CONFIG.replace(
    regions=regions,
    identity=identity,
    general_sr_strength=GENERAL_SR_STRENGTH,
    edge_sharpening=EDGE_SHARPENING,
    grain=GRAIN,
    skin_texture_smoothing=SKIN_TEXTURE_SMOOTHING,
    tonal_contrast=TONAL_CONTRAST,
    colour_balance=COLOR_BALANCE,
)
rows = [
    "| parameter | default | current | description | accepted | effect | warning |",
    "|---|---|---|---|---|---|---|",
]
for row in human_config.parameter_table(CONFIG):
    rows.append(
        "| "
        + " | ".join(
            str(row[key]) for key in ("parameter", "default", "current", "description", "accepted", "effect", "warning")
        )
        + " |"
    )
display(Markdown("\n".join(rows)))

## 8. Scene analysis
The package merges bundled Haar detections after validated manual boxes, assigns source-scale tiers and estimates protective region masks. Detection never proves identity.

## 9. Baseline reconstruction
Lanczos provides the original-derived baseline.

## 10. Pretrained SR
The Phase 2 presets select the fidelity network for conservative and the perceptual network for balanced, at native x4 then Lanczos-3 to the target. Learned detail is Inference, never recovered sensor information. Conservative is the final reconstruction starting point; balanced is a comparison candidate. Source-structure projection is enabled by default. The explicit lanczos_stand_in backend remains Interpolation. Weights, checksums, licences and commercial-use caveats are documented in docs/models.md. Run manifests and reports record the selected weights, model descriptions, tiling and seam checks. Missing learned dependencies or weights fail explicitly with guidance to select sr_backend: lanczos_stand_in; interpolation is never substituted silently. Identity failures retain the local source-derived Lanczos fallback.

## 11. Specialized reconstruction
Bounded region enhancement, identity consistency evaluation, PRD 11.6 conservative retry and original-derived fallback, and crowd checks run through the package API. Missing anatomy checks remain in the human-review list.

## 12. Tonal and optical processing
Tonal contrast (-0.5 to 0.5, default 0) changes the log-luminance slope around linear mid-grey 0.18, with gain bounded to [0.5, 2]. Colour balance uses linear RGB gains (each 0.5 to 1.5, default [1, 1, 1]). Both are classical Simulation stages before identity recovery, which can undo face changes. High strengths risk clipping and altered appearance. The colour stage reports mask-weighted skin CIEDE2000 before recovery, or explicitly unavailable when no skin samples exist. Skin smoothing runs before sharpening; optics are not applied.

## 13. Optional sensor simulation
Disabled. No synthetic sensor data is generated. Optional grain is Simulation: seeded achromatic independent output-pixel noise, amplitude 0 to 1 (default 0), bounded to 0.02 linear and scaled by sqrt(luminance). Face boxes are excluded; high values can obscure detail. Its stage reports Laplacian-luma energy before and after injection before identity recovery. Random energy is not recovered detail.

The following call executes sections 8 to 17 as one logged, reproducible package run. On failure, completed exports and diagnostic reports remain on disk; retry with a reduced quick target.

In [ ]:
try:
    RESULT = run_human_crowd(SOURCE, CONFIG, PROJECT, notebook_version=NOTEBOOK_VERSION)
except pipeline.StageError as error:
    print(f"Failed stage: {error.stage}; diagnostic manifest: {error.manifest_path}")
    print("Retry with RUNTIME_PRESET='quick' and a smaller QUICK_TARGET_MEGAPIXELS that still upscales the source")
    raise
MANIFEST = RESULT.manifest
print(f"run {RESULT.run_id}: {MANIFEST['status']} in {MANIFEST['total_seconds']:.1f} s")

## 14. Professional exports
Validated TIFF16 sRGB and half EXR linear Rec.709 masters include provenance. The review JPEG is for display.

In [ ]:
display(MANIFEST["exports"])
print(MANIFEST["review"]["path"])

## 15. Quality metrics
Per-tier consistency is not identity verification. Inspect unavailable metrics, failure events and required human review.

In [ ]:
display(MANIFEST["identity"])
display(MANIFEST["crowd"])
display(MANIFEST["failure_events"])
display(MANIFEST["source_consistency"])

## 16. Visual comparison
Inspect all seven PRD 18.3 comparisons and the mask overlay at native pixels. Labels reflect actual dimensions; quick outputs are not 50 MP.

In [ ]:
for name, figure in MANIFEST["comparisons"]["figures"].items():
    print(f"{name}: {figure['classification']} - {figure['description']}")
    display(Image(filename=figure["path"]))

## 17. Report generation
The report records per-tier identity metrics, review requirements, failure events, provenance, timing and stage peak memory.

In [ ]:
print(RESULT.manifest_path)
display(Markdown(RESULT.report_path.read_text()))

## 18. Download and archive
Archive reports separately from masters. Keep masters on Drive for Colab persistence.

In [ ]:
import shutil

archive = shutil.make_archive(str(RESULT.reports_dir), "zip", root_dir=RESULT.reports_dir)
print(f"Report archive: {archive}")
if IN_COLAB:
    from google.colab import files

    files.download(archive)